In [0]:
print("Project Atlas environment check")
print("Spark version:", spark.version)

display(spark.sql("SHOW CATALOGS"))

Project Atlas environment check
Spark version: 4.2.0


catalog
samples
system
workspace


In [0]:
display(
    spark.sql("""
        SELECT
            current_catalog() AS catalog,
            current_schema() AS schema
    """)
)

catalog,schema
workspace,default


In [0]:
display(spark.sql("SHOW SCHEMAS IN workspace"))
display(spark.sql("SELECT current_catalog(), current_schema()"))
display(spark.sql("SHOW TABLES IN workspace.default"))

databaseName
bronze
default
information_schema
retailpulse
training_demo


current_catalog(),current_schema()
workspace,default


database,tableName,isTemporary
default,dv_demo,false
default,emp_tmp_vw,false
default,movies,false
default,orders_partitioned,false
default,orders_unpartitioned,false
default,orders_zorder,false


In [0]:
from pyspark.sql import Row

test_df = spark.createDataFrame([
    Row(order_id=1, amount=125.50),
    Row(order_id=2, amount=240.00),
])

test_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.atlas_write_test"
)

display(spark.table("workspace.default.atlas_write_test"))

order_id,amount
1,125.5
2,240.0


In [0]:
display(
    spark.sql("""
        DESCRIBE DETAIL workspace.default.atlas_write_test
    """)
)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,8413a2e9-5018-419a-a4d3-8ab33b736dbd,workspace.default.atlas_write_test,null,,2026-10-24T12:27:35.494Z,2026-10-24T12:27:45.000Z,List(),List(),1,1098,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.atlas_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.atlas_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.atlas_gold")

display(spark.sql("SHOW SCHEMAS IN workspace"))

databaseName
atlas_bronze
atlas_gold
atlas_silver
bronze
default
information_schema
retailpulse
training_demo


In [0]:
path = "s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(path)
)

print("Rows:", df.count())
display(df.limit(5))

Rows: 99441


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z


In [0]:
table_name = "workspace.atlas_bronze.orders"

exists = spark.catalog.tableExists(table_name)
print("Table already exists:", exists)

Table already exists: False


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.atlas_ops;

CREATE TABLE IF NOT EXISTS workspace.atlas_ops.pipeline_run_audit (
    run_id STRING,
    batch_id STRING,
    pipeline_name STRING,
    stage STRING,
    status STRING,
    started_at TIMESTAMP,
    completed_at TIMESTAMP,
    source_file_count INT,
    records_processed BIGINT,
    error_message STRING,
    created_at TIMESTAMP
)
USING DELTA;

In [0]:
%sql
DESCRIBE TABLE workspace.atlas_ops.pipeline_run_audit;

col_name,data_type,comment
run_id,string,null
batch_id,string,null
pipeline_name,string,null
stage,string,null
status,string,null
started_at,timestamp,null
completed_at,timestamp,null
source_file_count,int,null
records_processed,bigint,null
error_message,string,null


In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F
import traceback

def write_audit(
    run_id,
    batch_id,
    stage,
    status,
    started_at,
    source_file_count=0,
    records_processed=0,
    error_message=None
):
    completed_at = datetime.now(timezone.utc).replace(tzinfo=None)

    audit_row = [(
        run_id,
        batch_id,
        pipeline_name,
        stage,
        status,
        started_at,
        completed_at,
        int(source_file_count),
        int(records_processed),
        error_message,
        completed_at
    )]

    columns = [
        "run_id", "batch_id", "pipeline_name", "stage", "status",
        "started_at", "completed_at", "source_file_count",
        "records_processed", "error_message", "created_at"
    ]

    (
        spark.createDataFrame(audit_row, columns)
        .write.format("delta")
        .mode("append")
        .saveAsTable("workspace.atlas_ops.pipeline_run_audit")
    )

In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid
import traceback

source_folders = {
    "orders": (
        "s3://project-atlas-olist-20261024-180312/raw/olist/orders/"
    ),
    "order_items": (
        "s3://project-atlas-olist-20261024-180312/raw/olist/order_items/"
    ),
    "products": (
        "s3://project-atlas-olist-20261024-180312/raw/olist/products/"
    )
}

target_tables = {
    "orders": "workspace.atlas_bronze.orders",
    "order_items": "workspace.atlas_bronze.order_items",
    "products": "workspace.atlas_bronze.products"
}

pipeline_run_id = str(uuid.uuid4())
pipeline_name = "Project Atlas - Retail Data Pipeline"
bronze_started_at = datetime.now(timezone.utc).replace(tzinfo=None)
batch_id = None
source_file_count = 0
records_processed = 0


from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, IntegerType, LongType
)

def write_audit(
    run_id,
    batch_id,
    stage,
    status,
    started_at,
    source_file_count=0,
    records_processed=0,
    error_message=None
):
    completed_at = datetime.now(timezone.utc).replace(tzinfo=None)

    audit_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("batch_id", StringType(), True),
        StructField("pipeline_name", StringType(), True),
        StructField("stage", StringType(), True),
        StructField("status", StringType(), True),
        StructField("started_at", TimestampType(), True),
        StructField("completed_at", TimestampType(), True),
        StructField("source_file_count", IntegerType(), True),
        StructField("records_processed", LongType(), True),
        StructField("error_message", StringType(), True),
        StructField("created_at", TimestampType(), True)
    ])

    audit_row = [(
        run_id,
        batch_id,
        pipeline_name,
        stage,
        status,
        started_at,
        completed_at,
        int(source_file_count),
        int(records_processed),
        error_message,
        completed_at
    )]

    (
        spark.createDataFrame(audit_row, schema=audit_schema)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable("workspace.atlas_ops.pipeline_run_audit")
    )


try:
    current_sources = {}

    for dataset, folder in source_folders.items():
        csv_files = [
            file_info.path
            for file_info in dbutils.fs.ls(folder)
            if file_info.name.lower().endswith(".csv")
        ]

        if not csv_files:
            raise ValueError(
                f"No CSV files found for dataset '{dataset}' in {folder}"
            )

        current_sources[dataset] = sorted(csv_files)

        print(f"{dataset}: {len(csv_files)} CSV file(s) found")
        for path in current_sources[dataset]:
            print(f"  {path}")

    source_file_count = sum(
        len(paths) for paths in current_sources.values()
    )

    batch_id = (
        "atlas_refresh_"
        + spark.sql("""
            SELECT date_format(
                current_timestamp(),
                'yyyyMMdd_HHmmss'
            ) AS id
        """).first()["id"]
    )

    for dataset, paths in current_sources.items():
        raw_df = (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(paths)
        )

        bronze_df = (
            raw_df
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.col("_metadata.file_path"))
            .withColumn("_batch_id", F.lit(batch_id))
        )

        row_count = bronze_df.count()

        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_tables[dataset])
        )

        records_processed += row_count

        print(
            f"Rebuilt {target_tables[dataset]} "
            f"from {len(paths)} source file(s), "
            f"{row_count} records"
        )

    write_audit(
        run_id=pipeline_run_id,
        batch_id=batch_id,
        stage="BRONZE",
        status="SUCCESS",
        started_at=bronze_started_at,
        source_file_count=source_file_count,
        records_processed=records_processed
    )

    print(f"Bronze full refresh completed. Batch ID: {batch_id}")
    print(f"Pipeline run ID: {pipeline_run_id}")
    print(f"Total records processed: {records_processed}")

except Exception as e:
    try:
        write_audit(
            run_id=pipeline_run_id,
            batch_id=batch_id,
            stage="BRONZE",
            status="FAILED",
            started_at=bronze_started_at,
            source_file_count=source_file_count,
            records_processed=records_processed,
            error_message=traceback.format_exc()[-8000:]
        )
    except Exception as audit_error:
        print(f"Failed to write audit record: {audit_error}")

    raise


orders: 1 CSV file(s) found
  s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv
order_items: 1 CSV file(s) found
  s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv
products: 1 CSV file(s) found
  s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv
Rebuilt workspace.atlas_bronze.orders from 1 source file(s), 99441 records
Rebuilt workspace.atlas_bronze.order_items from 1 source file(s), 112650 records
Rebuilt workspace.atlas_bronze.products from 1 source file(s), 32951 records
Bronze full refresh completed. Batch ID: atlas_refresh_20261024_224621
Pipeline run ID: 91feab5c-0ad2-4482-b709-1c50f5c3d449
Total records processed: 245042


In [0]:
%sql
SELECT *
FROM workspace.atlas_ops.pipeline_run_audit
ORDER BY created_at DESC;

run_id,batch_id,pipeline_name,stage,status,started_at,completed_at,source_file_count,records_processed,error_message,created_at
91feab5c-0ad2-4482-b709-1c50f5c3d449,atlas_refresh_20261024_224621,Project Atlas - Retail Data Pipeline,BRONZE,SUCCESS,2026-10-24T22:46:17.230Z,2026-10-24T22:47:08.819Z,3,245042,null,2026-10-24T22:47:08.819Z
e3aa3da9-856a-4519-9e56-c0d8fec0f6a4,atlas_refresh_20261024_224420,Project Atlas - Retail Data Pipeline,BRONZE,FAILED,2026-10-24T22:44:16.119Z,2026-10-24T22:45:02.895Z,3,245042,"Traceback (most recent call last): File ""/home/spark-5498c5c9-0975-4fd0-a81f-5a/.ipykernel/63/command-5930624001403202-2838441095"", line 148, in write_audit( File ""/home/spark-5498c5c9-0975-4fd0-a81f-5a/.ipykernel/63/command-5930624001403202-2838441095"", line 73, in write_audit spark.createDataFrame(audit_row, columns) File ""/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py"", line 820, in createDataFrame raise PySparkValueError( pyspark.errors.exceptions.base.PySparkValueError: [CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.",2026-10-24T22:45:02.895Z


In [0]:
from datetime import datetime, timezone
import uuid

pipeline_run_id = str(uuid.uuid4())
pipeline_name = "Project Atlas - Retail Data Pipeline"

bronze_started_at = datetime.now(timezone.utc).replace(tzinfo=None)

print("Pipeline run ID:", pipeline_run_id)
print("Bronze stage started at:", bronze_started_at)

Pipeline run ID: d8079445-665a-4cf6-b6d5-9ff69db7b925
Bronze stage started at: 2026-10-24 22:39:21.541562


In [0]:
from datetime import datetime, timezone

processed_metadata = []

for dataset, paths in current_sources.items():
    for path in paths:
        parent_folder = path.rsplit("/", 1)[0] + "/"

        file_info = next(
            (
                f for f in dbutils.fs.ls(parent_folder)
                if f.path == path
            ),
            None
        )

        if file_info is None:
            raise FileNotFoundError(
                f"Source file disappeared during ingestion: {path}"
            )

        processed_metadata.append({
            "source_path": path,
            "file_size_bytes": int(file_info.size),
            "last_modified": datetime.fromtimestamp(
                file_info.modificationTime / 1000,
                tz=timezone.utc
            ).replace(tzinfo=None)
        })

print(f"Source files tracked for this run: {len(processed_metadata)}")


Source files tracked for this run: 3


In [0]:
expected_counts = {
    "orders" : 99441,
    "order_items" : 112650,
    "products" : 32951
}

for name, expected in expected_counts.items():
    table_name = f"workspace.atlas_bronze.{name}"
    df = spark.table(table_name)
    actual = df.count()

    print(
        f"{name}: expected={expected}, "
        f"actual={actual}, "
        f"PASS={actual == expected}"
    )

orders: expected=99441, actual=99441, PASS=True
order_items: expected=112650, actual=112650, PASS=True
products: expected=32951, actual=32951, PASS=True


In [0]:
from pyspark.sql import functions as F

for name in ["orders", "order_items", "products"]:
    df = spark.table(f"workspace.atlas_bronze.{name}")
    

    print(f"\n---{name}---")
    print("Columns: ", df.columns)

    display(
        df.select(
            "_ingested_at",
            "_source_file",
            "_batch_id"
        ).limit(1)
    )


---orders---
Columns:  ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', '_ingested_at', '_source_file', '_batch_id']


_ingested_at,_source_file,_batch_id
2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947



---order_items---
Columns:  ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', '_ingested_at', '_source_file', '_batch_id']


_ingested_at,_source_file,_batch_id
2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947



---products---
Columns:  ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', '_ingested_at', '_source_file', '_batch_id']


_ingested_at,_source_file,_batch_id
2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947


In [0]:
orders_bronze = spark.table("workspace.atlas_bronze.orders")

orders_bronze.printSchema()
display(orders_bronze.limit(5))

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_ingested_at,_source_file,_batch_id
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947


In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone
import traceback

silver_started_at = datetime.now(timezone.utc).replace(tzinfo=None)

try:
    orders_bronze = spark.table("workspace.atlas_bronze.orders")

    timestamp_cols = [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]

    orders_silver = orders_bronze

    for col_name in timestamp_cols:
        orders_silver = orders_silver.withColumn(
            col_name,
            F.to_timestamp(F.col(col_name))
        )

    orders_silver = orders_silver.withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    )

    orders_silver = (
        orders_silver
        .withColumn(
            "_dq_missing_order_id",
            F.col("order_id").isNull() | (F.trim(F.col("order_id")) == "")
        )
        .withColumn(
            "_dq_invalid_timestamp_order",
            F.col("order_purchase_timestamp").isNull()
        )
        .withColumn(
            "_dq_delivery_before_purchase",
            F.col("order_delivered_customer_date").isNotNull()
            & F.col("order_purchase_timestamp").isNotNull()
            & (
                F.col("order_delivered_customer_date")
                < F.col("order_purchase_timestamp")
            )
        )
    )

    orders_count = orders_silver.count()

    (
        orders_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.atlas_silver.orders")
    )

    write_audit(
        run_id=pipeline_run_id,
        batch_id=batch_id,
        stage="SILVER_ORDERS",
        status="SUCCESS",
        started_at=silver_started_at,
        records_processed=orders_count
    )

    print("Silver orders table created.")
    print(f"Records processed: {orders_count}")

except Exception:
    try:
        write_audit(
            run_id=pipeline_run_id,
            batch_id=batch_id,
            stage="SILVER_ORDERS",
            status="FAILED",
            started_at=silver_started_at,
            error_message=traceback.format_exc()[-8000:]
        )
    except Exception as audit_error:
        print(f"Failed to write audit record: {audit_error}")

    raise


Silver orders table created.
Records processed: 99441


In [0]:
orders_silver = spark.table("workspace.atlas_silver.orders")

print("Bronze rows:", spark.table("workspace.atlas_bronze.orders").count())
print("Silver rows:", orders_silver.count())

print(
    "Missing order IDs:",
    orders_silver.filter(F.col("_dq_missing_order_id")).count()
)

print(
    "Invalid purchase timestamps:",
    orders_silver.filter(F.col("_dq_invalid_timestamp_order")).count()
)

print(
    "Delivery before purchase:",
    orders_silver.filter(F.col("_dq_delivery_before_purchase")).count()
)

orders_silver.printSchema()
display(orders_silver.limit(10))

Bronze rows: 99441
Silver rows: 99441
Missing order IDs: 0
Invalid purchase timestamps: 0
Delivery before purchase: 0
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _dq_missing_order_id: boolean (nullable = true)
 |-- _dq_invalid_timestamp_order: boolean (nullable = true)
 |-- _dq_delivery_before_purchase: boolean (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_ingested_at,_source_file,_batch_id,_dq_missing_order_id,_dq_invalid_timestamp_order,_dq_delivery_before_purchase
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947,false,false,false
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947,false,false,false
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947,false,false,false
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947,false,false,false
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947,false,false,false
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947,false,false,false
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947,false,false,false
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947,false,false,false
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947,false,false,false
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,2026-10-24T22:20:00.346Z,s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,atlas_refresh_20261024_221947,false,false,false


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from datetime import datetime, timezone
import traceback

items_started_at = datetime.now(timezone.utc).replace(tzinfo=None)

try:
    items = spark.table("workspace.atlas_bronze.order_items")

    items_silver = (
        items
        .withColumn("order_id", F.trim(F.col("order_id")))
        .withColumn("product_id", F.trim(F.col("product_id")))
        .withColumn("seller_id", F.trim(F.col("seller_id")))
        .withColumn("order_item_id", F.col("order_item_id").cast("int"))
        .withColumn("price", F.col("price").cast(DecimalType(12, 2)))
        .withColumn("freight_value", F.col("freight_value").cast(DecimalType(12, 2)))
        .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
    )

    items_silver = (
        items_silver
        .withColumn(
            "_dq_missing_order_id",
            F.col("order_id").isNull() | (F.col("order_id") == "")
        )
        .withColumn(
            "_dq_missing_product_id",
            F.col("product_id").isNull() | (F.col("product_id") == "")
        )
        .withColumn(
            "_dq_invalid_item_id",
            F.col("order_item_id").isNull() | (F.col("order_item_id") <= 0)
        )
        .withColumn(
            "_dq_invalid_price",
            F.col("price").isNull() | (F.col("price") < 0)
        )
        .withColumn(
            "_dq_invalid_freight",
            F.col("freight_value").isNull() | (F.col("freight_value") < 0)
        )
    )

    items_count = items_silver.count()

    (
        items_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.atlas_silver.order_items")
    )

    write_audit(
        run_id=pipeline_run_id,
        batch_id=batch_id,
        stage="SILVER_ORDER_ITEMS",
        status="SUCCESS",
        started_at=items_started_at,
        records_processed=items_count
    )

    print("Silver order_items table created.")
    print(f"Records processed: {items_count}")

except Exception:
    try:
        write_audit(
            run_id=pipeline_run_id,
            batch_id=batch_id,
            stage="SILVER_ORDER_ITEMS",
            status="FAILED",
            started_at=items_started_at,
            error_message=traceback.format_exc()[-8000:]
        )
    except Exception as audit_error:
        print(f"Failed to write audit record: {audit_error}")

    raise


Silver order_items table created.
Records processed: 112650


In [0]:
items_silver = spark.table("workspace.atlas_silver.order_items")

print("Bronze rows:", spark.table("workspace.atlas_bronze.order_items").count())
print("Silver rows:", items_silver.count())

quality_flags = [
    "_dq_missing_order_id",
    "_dq_missing_product_id",
    "_dq_invalid_item_id",
    "_dq_invalid_price",
    "_dq_invalid_freight"
]

for flag in quality_flags:
    print(flag, items_silver.filter(F.col(flag)).count())

items_silver.printSchema()
display(items_silver.limit(10))

Bronze rows: 112650
Silver rows: 112650
_dq_missing_order_id 0
_dq_missing_product_id 0
_dq_invalid_item_id 0
_dq_invalid_price 0
_dq_invalid_freight 0
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _dq_missing_order_id: boolean (nullable = true)
 |-- _dq_missing_product_id: boolean (nullable = true)
 |-- _dq_invalid_item_id: boolean (nullable = true)
 |-- _dq_invalid_price: boolean (nullable = true)
 |-- _dq_invalid_freight: boolean (nullable = true)



order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,_ingested_at,_source_file,_batch_id,_dq_missing_order_id,_dq_missing_product_id,_dq_invalid_item_id,_dq_invalid_price,_dq_invalid_freight
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.90,13.29,2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947,false,false,false,false,false
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.90,19.93,2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947,false,false,false,false,false
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.00,17.87,2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947,false,false,false,false,false
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947,false,false,false,false,false
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.90,18.14,2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947,false,false,false,false,false
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.90,12.69,2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947,false,false,false,false,false
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.90,11.85,2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947,false,false,false,false,false
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.00,70.75,2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947,false,false,false,false,false
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65,2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947,false,false,false,false,false
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.40,2026-10-24T22:20:21.868Z,s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,atlas_refresh_20261024_221947,false,false,false,false,false


In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone
import traceback

products_started_at = datetime.now(timezone.utc).replace(tzinfo=None)

try:
    products = spark.table("workspace.atlas_bronze.products")

    products_silver = (
        products
        .withColumn("product_id", F.trim(F.col("product_id")))
        .withColumn(
            "product_category_name",
            F.when(
                F.col("product_category_name").isNull()
                | (F.trim(F.col("product_category_name")) == ""),
                F.lit("unknown")
            ).otherwise(F.lower(F.trim(F.col("product_category_name"))))
        )
        .withColumn("product_name_lenght", F.col("product_name_lenght").cast("int"))
        .withColumn("product_description_lenght", F.col("product_description_lenght").cast("int"))
        .withColumn("product_photos_qty", F.col("product_photos_qty").cast("int"))
        .withColumn("product_weight_g", F.col("product_weight_g").cast("int"))
        .withColumn("product_length_cm", F.col("product_length_cm").cast("int"))
        .withColumn("product_height_cm", F.col("product_height_cm").cast("int"))
        .withColumn("product_width_cm", F.col("product_width_cm").cast("int"))
    )

    products_silver = (
        products_silver
        .withColumn(
            "_dq_missing_product_id",
            F.col("product_id").isNull() | (F.col("product_id") == "")
        )
        .withColumn(
            "_dq_invalid_weight",
            F.col("product_weight_g").isNotNull()
            & (F.col("product_weight_g") < 0)
        )
        .withColumn(
            "_dq_invalid_dimensions",
            (
                (F.col("product_length_cm") < 0)
                | (F.col("product_height_cm") < 0)
                | (F.col("product_width_cm") < 0)
            )
        )
    )

    products_count = products_silver.count()

    (
        products_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.atlas_silver.products")
    )

    write_audit(
        run_id=pipeline_run_id,
        batch_id=batch_id,
        stage="SILVER_PRODUCTS",
        status="SUCCESS",
        started_at=products_started_at,
        records_processed=products_count
    )

    print("Silver products table created.")
    print(f"Records processed: {products_count}")

except Exception:
    try:
        write_audit(
            run_id=pipeline_run_id,
            batch_id=batch_id,
            stage="SILVER_PRODUCTS",
            status="FAILED",
            started_at=products_started_at,
            error_message=traceback.format_exc()[-8000:]
        )
    except Exception as audit_error:
        print(f"Failed to write audit record: {audit_error}")

    raise


Silver products table created.
Records processed: 32951


In [0]:
products_silver = spark.table("workspace.atlas_silver.products")

print("Bronze rows:", spark.table("workspace.atlas_bronze.products").count())
print("Silver rows:", products_silver.count())
print(
    "Missing product IDs:",
    products_silver.filter(F.col("_dq_missing_product_id")).count()
)

products_silver.printSchema()
display(products_silver.limit(10))

Bronze rows: 32951
Silver rows: 32951
Missing product IDs: 0
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- _dq_missing_product_id: boolean (nullable = true)
 |-- _dq_invalid_weight: boolean (nullable = true)
 |-- _dq_invalid_dimensions: boolean (nullable = true)



product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_ingested_at,_source_file,_batch_id,_dq_missing_product_id,_dq_invalid_weight,_dq_invalid_dimensions
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947,false,false,false
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947,false,false,false
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947,false,false,false
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947,false,false,false
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947,false,false,false
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11,2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947,false,false,false
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44,2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947,false,false,false
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40,2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947,false,false,false
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17,2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947,false,false,false
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12,2026-10-24T22:20:35.496Z,s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,atlas_refresh_20261024_221947,false,false,false


In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone
import traceback

enriched_started_at = datetime.now(timezone.utc).replace(tzinfo=None)

try:
    orders = spark.table("workspace.atlas_silver.orders")
    items = spark.table("workspace.atlas_silver.order_items")
    products = spark.table("workspace.atlas_silver.products")

    orders_selected = orders.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )

    items_selected = items.select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    )

    products_selected = products.select(
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )

    enriched = (
        items_selected
        .join(orders_selected, on="order_id", how="left")
        .join(products_selected, on="product_id", how="left")
        .withColumn(
            "_dq_missing_order_match",
            F.col("customer_id").isNull()
        )
        .withColumn(
            "_dq_missing_product_match",
            F.col("product_id").isNotNull()
            & F.col("product_category_name").isNull()
        )
    )

    enriched_count = enriched.count()

    (
        enriched.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.atlas_silver.order_item_enriched")
    )

    write_audit(
        run_id=pipeline_run_id,
        batch_id=batch_id,
        stage="SILVER_ORDER_ITEM_ENRICHED",
        status="SUCCESS",
        started_at=enriched_started_at,
        records_processed=enriched_count
    )

    print("Enriched Silver table created.")
    print(f"Records processed: {enriched_count}")

except Exception:
    try:
        write_audit(
            run_id=pipeline_run_id,
            batch_id=batch_id,
            stage="SILVER_ORDER_ITEM_ENRICHED",
            status="FAILED",
            started_at=enriched_started_at,
            error_message=traceback.format_exc()[-8000:]
        )
    except Exception as audit_error:
        print(f"Failed to write audit record: {audit_error}")

    raise


Enriched Silver table created.
Records processed: 112650


In [0]:
%sql

SELECT
    stage,
    status,
    COUNT(*) AS audit_records,
    SUM(records_processed) AS total_records_processed,
    MAX(created_at) AS latest_run
FROM workspace.atlas_ops.pipeline_run_audit
GROUP BY stage, status
ORDER BY latest_run DESC;


stage,status,audit_records,total_records_processed,latest_run
SILVER_ORDER_ITEM_ENRICHED,SUCCESS,1,112650,2026-10-24T22:53:20.691Z
SILVER_PRODUCTS,SUCCESS,2,65902,2026-10-24T22:52:45.950Z
SILVER_ORDER_ITEMS,SUCCESS,2,225300,2026-10-24T22:52:32.590Z
SILVER_ORDERS,SUCCESS,2,198882,2026-10-24T22:52:15.394Z
BRONZE,SUCCESS,1,245042,2026-10-24T22:47:08.819Z
BRONZE,FAILED,1,245042,2026-10-24T22:45:02.895Z


In [0]:
items_count = spark.table(
    "workspace.atlas_silver.order_items"
).count()

enriched_df = spark.table(
    "workspace.atlas_silver.order_item_enriched"
)

enriched_count = enriched_df.count()

print("Order items:", items_count)
print("Enriched rows:", enriched_count)
print("Row count preserved:", items_count == enriched_count)

print(
    "Missing order matches:",
    enriched_df.filter(F.col("_dq_missing_order_match")).count()
)

print(
    "Missing product matches:",
    enriched_df.filter(F.col("_dq_missing_product_match")).count()
)

display(enriched_df.limit(10))

Order items: 112650
Enriched rows: 112650
Row count preserved: True
Missing order matches: 0
Missing product matches: 0


product_id,order_id,order_item_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_dq_missing_order_match,_dq_missing_product_match
4535b0e1091c278dfd193e5a1d63b39f,0005f50442cb953dcd1d21e1fb923495,1,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.40,351d3cb2cee3c7fd0af6616c82df21d3,delivered,2018-07-02T13:59:39.000Z,2018-07-02T14:10:56.000Z,2018-07-03T14:25:00.000Z,2018-07-04T17:28:31.000Z,2018-07-23T00:00:00.000Z,livros_tecnicos,850,29,3,21,false,false
f177554ea93259a5b282f24e33f65ab6,00063b381e2406b52ad429470734ebd5,1,8602a61d680a10a82cceeeda0d99ea3d,2018-07-31T17:30:39.000Z,45.00,12.98,6a899e55865de6549a58d2c6845e5604,delivered,2018-07-27T17:21:27.000Z,2018-07-27T18:00:06.000Z,2018-07-30T14:52:00.000Z,2018-08-07T13:56:52.000Z,2018-08-07T00:00:00.000Z,fashion_bolsas_e_acessorios,200,16,16,11,false,false
5d7c23067ed3fc8c6e699b9373d5890b,001021efaa8636c29475e7734483457d,1,6560211a19b47992c3666cc44a7e94c0,2018-03-05T09:35:41.000Z,49.00,15.10,2dfbf74859104caf100df3720a1d833d,delivered,2018-02-27T09:27:14.000Z,2018-02-27T09:35:41.000Z,2018-02-27T21:43:46.000Z,2018-03-08T20:06:33.000Z,2018-03-23T00:00:00.000Z,fashion_bolsas_e_acessorios,250,16,2,20,false,false
777d2e438a1b645f3aec9bd57e92672c,001dbc16dc51075e987543d23a0507c7,1,4a3ca9315b744ce9f8e9374361493884,2017-02-01T13:17:57.000Z,69.90,18.00,698a74f33469466fa4172e829505d1c6,delivered,2017-01-28T13:17:57.000Z,2017-01-28T13:32:16.000Z,2017-02-01T15:59:46.000Z,2017-02-13T13:17:47.000Z,2017-03-20T00:00:00.000Z,cama_mesa_banho,950,45,15,35,false,false
71c89e9d8fa274b017279aa322cd0e19,00276d5c3491fbf55305e26891040df9,1,8f78f0903005064036736c7173a5c2ed,2018-02-27T11:55:57.000Z,44.90,23.22,54686c41e86854e35879efbd1c2cab1b,delivered,2018-02-13T11:47:21.000Z,2018-02-13T12:55:57.000Z,2018-02-20T23:19:10.000Z,2018-03-16T19:57:52.000Z,2018-03-22T00:00:00.000Z,moveis_decoracao,2750,70,5,15,false,false
84904413b82eee333a3f79137e8d197e,003423b755b562962a6225a8de40d12e,1,23c38debaffe4a25a30fdbd9b586a13f,2018-07-18T04:25:09.000Z,232.75,28.58,18f1bb6325d50619d5c13b7a25b869fd,delivered,2018-07-08T11:28:17.000Z,2018-07-10T04:25:09.000Z,2018-07-13T15:00:00.000Z,2018-07-24T21:33:38.000Z,2018-08-08T00:00:00.000Z,ferramentas_jardim,1600,18,22,17,false,false
452f66a0f164cac57802e2cea93188ac,003a94f778ef8cfd50247c8c1b582257,1,f80edd2c5aaa505cc4b0a3b219abf4b8,2018-08-07T15:05:14.000Z,39.90,18.08,da108bc410acffb1400cf9caea65a332,delivered,2018-08-03T14:52:31.000Z,2018-08-03T15:05:14.000Z,2018-08-06T14:12:00.000Z,2018-08-24T00:18:38.000Z,2018-08-15T00:00:00.000Z,construcao_ferramentas_construcao,350,16,61,16,false,false
d0fb1e667e989933a80444f93da833c0,004345d16a1ab2c21962992c721c8643,1,0be8ff43f22e456b4e0371b2245e4d01,2018-07-10T07:31:33.000Z,37.90,15.37,3b723b649feb48f3cfbb9c728efb3d7d,delivered,2018-07-04T06:45:05.000Z,2018-07-05T16:09:18.000Z,2018-07-05T14:08:00.000Z,2018-07-11T17:26:37.000Z,2018-07-30T00:00:00.000Z,utilidades_domesticas,525,18,19,16,false,false
ff6caf9340512b8bf6d2a2a6df032cfa,0046e1d57f4c07c8c92ab26be8c3dfc0,1,38e6dada03429a47197d5d584d793b41,2017-10-02T15:49:17.000Z,7.79,7.78,ae55e228ded796581eccc67045516023,delivered,2017-09-26T15:24:03.000Z,2017-09-26T15:49:17.000Z,2017-09-28T18:42:42.000Z,2017-09-29T18:29:50.000Z,2017-10-11T00:00:00.000Z,unknown,200,16,5,12,false,false
0c753fe6a58dfa7a27bda5de76e779c3,004eab0fd8f28adaf8d488976f77febe,1,7586919161935337bf6b6d7ff5779648,2017-08-08T15:45:17.000Z,21.50,11.98,c476ddfbabfa4624603e0b7f8e245057,delivered,2017-08-02T15:32:46.000Z,2017-08-02T15:45:17.000Z,2017-08-02T20:32:44.000Z,2017-08-03T18:17:47.000Z,2017-08-15T00:00:00.000Z,esporte_lazer,1338,23,12,14,false,false


In [0]:
from pyspark.sql import functions as F

tables = {
    "orders": ("workspace.atlas_bronze.orders",
               "workspace.atlas_silver.orders", 99441),
    "order_items": ("workspace.atlas_bronze.order_items",
                    "workspace.atlas_silver.order_items", 112650),
    "products": ("workspace.atlas_bronze.products",
                 "workspace.atlas_silver.products", 32951),
}

for name, (bronze_name, silver_name, expected) in tables.items():
    bronze_count = spark.table(bronze_name).count()
    silver_count = spark.table(silver_name).count()

    print(
        f"{name}: Bronze={bronze_count}, Silver={silver_count}, "
        f"Expected={expected}, "
        f"PASS={bronze_count == silver_count == expected}"
    )

enriched = spark.table("workspace.atlas_silver.order_item_enriched")
items_count = spark.table("workspace.atlas_silver.order_items").count()
enriched_count = enriched.count()

print(
    f"Enriched: order_items={items_count}, "
    f"enriched={enriched_count}, "
    f"PASS={items_count == enriched_count}"
)

print(
    "Duplicate order-item keys:",
    enriched.groupBy("order_id", "order_item_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
)

print(
    "Missing order matches:",
    enriched.filter(F.col("_dq_missing_order_match")).count()
)

print(
    "Missing product matches:",
    enriched.filter(F.col("_dq_missing_product_match")).count()
)

orders: Bronze=99441, Silver=99441, Expected=99441, PASS=True
order_items: Bronze=112650, Silver=112650, Expected=112650, PASS=True
products: Bronze=32951, Silver=32951, Expected=32951, PASS=True
Enriched: order_items=112650, enriched=112650, PASS=True
Duplicate order-item keys: 0
Missing order matches: 0
Missing product matches: 0


In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone
import traceback

daily_sales_started_at = datetime.now(timezone.utc).replace(tzinfo=None)

try:
    orders = spark.table("workspace.atlas_silver.orders")

    daily_orders = (
        orders
        .withColumn(
            "purchase_date",
            F.to_date("order_purchase_timestamp")
        )
        .groupBy("purchase_date")
        .agg(
            F.countDistinct("order_id").alias("order_count"),
            F.countDistinct("customer_id").alias("customer_count"),
            F.countDistinct(
                F.when(
                    F.col("order_status") == "delivered",
                    F.col("order_id")
                )
            ).alias("delivered_orders"),
            F.countDistinct(
                F.when(
                    F.col("order_status") == "canceled",
                    F.col("order_id")
                )
            ).alias("cancelled_orders")
        )
    )

    enriched = spark.table("workspace.atlas_silver.order_item_enriched")

    daily_items = (
        enriched
        .withColumn(
            "purchase_date",
            F.to_date("order_purchase_timestamp")
        )
        .groupBy("purchase_date")
        .agg(
            F.count("*").alias("items_sold"),
            F.sum("price").alias("items_sales_value"),
            F.sum("freight_value").alias("freight_value")
        )
    )

    daily_sales = (
        daily_orders
        .join(daily_items, on="purchase_date", how="left")
        .fillna({
            "items_sold": 0,
            "items_sales_value": 0,
            "freight_value": 0
        })
    )

    daily_sales_count = daily_sales.count()

    (
        daily_sales.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.atlas_gold.daily_sales")
    )

    write_audit(
        run_id=pipeline_run_id,
        batch_id=batch_id,
        stage="GOLD_DAILY_SALES",
        status="SUCCESS",
        started_at=daily_sales_started_at,
        records_processed=daily_sales_count
    )

    print("Rebuilt workspace.atlas_gold.daily_sales")
    print(f"Records processed: {daily_sales_count}")

except Exception:
    try:
        write_audit(
            run_id=pipeline_run_id,
            batch_id=batch_id,
            stage="GOLD_DAILY_SALES",
            status="FAILED",
            started_at=daily_sales_started_at,
            error_message=traceback.format_exc()[-8000:]
        )
    except Exception as audit_error:
        print(f"Failed to write audit record: {audit_error}")

    raise


Rebuilt workspace.atlas_gold.daily_sales
Records processed: 634


In [0]:
from pyspark.sql import functions as F

daily = spark.table("workspace.atlas_gold.daily_sales")
source = spark.table("workspace.atlas_silver.order_item_enriched")

print("Gold columns:", daily.columns)

daily.select(
    F.min("purchase_date").alias("first_date"),
    F.max("purchase_date").alias("last_date"),
    F.sum("order_count").alias("daily_order_total"),
    F.sum("items_sold").alias("total_items"),
    F.sum("items_sales_value").alias("total_item_sales"),
    F.sum("freight_value").alias("total_freight")
).show(truncate=False)

source_sales = source.agg(F.sum("price")).first()[0]
gold_sales = daily.agg(F.sum("items_sales_value")).first()[0]

source_items = source.count()
gold_items = daily.agg(F.sum("items_sold")).first()[0]

source_orders = source.select("order_id").distinct().count()
gold_orders = daily.agg(F.sum("order_count")).first()[0]

print("Source items:", source_items, "| Gold items:", gold_items)
print("Items reconcile:", source_items == gold_items)

print("Source distinct orders:", source_orders, "| Gold daily orders:", gold_orders)
print("Orders reconcile:", source_orders == gold_orders)

print("Source item sales:", source_sales, "| Gold item sales:", gold_sales)
print("Sales reconcile:", source_sales == gold_sales)

display(daily.orderBy("purchase_date").limit(15))

Gold columns: ['purchase_date', 'order_count', 'customer_count', 'delivered_orders', 'cancelled_orders', 'items_sold', 'items_sales_value', 'freight_value']
+----------+----------+-----------------+-----------+----------------+-------------+
|first_date|last_date |daily_order_total|total_items|total_item_sales|total_freight|
+----------+----------+-----------------+-----------+----------------+-------------+
|2016-09-04|2018-10-17|99441            |112650     |13591643.70     |2251909.54   |
+----------+----------+-----------------+-----------+----------------+-------------+

Source items: 112650 | Gold items: 112650
Items reconcile: True
Source distinct orders: 98666 | Gold daily orders: 99441
Orders reconcile: False
Source item sales: 13591643.70 | Gold item sales: 13591643.70
Sales reconcile: True


purchase_date,order_count,customer_count,delivered_orders,cancelled_orders,items_sold,items_sales_value,freight_value
2016-09-04,1,1,0,0,2,72.89,63.34
2016-09-05,1,1,0,1,1,59.50,15.56
2016-09-13,1,1,0,1,0,0.00,0.00
2016-09-15,1,1,1,0,3,134.97,8.49
2016-10-02,1,1,0,1,1,100.00,9.34
2016-10-03,8,8,7,1,8,463.48,131.66
2016-10-04,63,63,54,5,69,9940.96,1354.52
2016-10-05,47,47,35,2,56,8343.25,1302.69
2016-10-06,51,51,41,3,58,7960.51,1170.72
2016-10-07,46,46,38,2,52,7228.05,907.92


In [0]:

from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime, timezone
import traceback

category_started_at = datetime.now(timezone.utc).replace(tzinfo=None)

try:
    enriched = spark.table("workspace.atlas_silver.order_item_enriched")

    category_performance = (
        enriched
        .withColumn(
            "category",
            F.coalesce(
                F.col("product_category_name"),
                F.lit("unknown")
            )
        )
        .groupBy("category")
        .agg(
            F.count("*").alias("items_sold"),
            F.countDistinct("order_id").alias("order_count"),
            F.sum("price").alias("items_sales_value"),
            F.sum("freight_value").alias("freight_value"),
            F.avg("price").alias("average_item_price")
        )
        .withColumn(
            "sales_rank",
            F.dense_rank().over(
                Window.orderBy(
                    F.col("items_sales_value").desc_nulls_last()
                )
            )
        )
    )

    category_count = category_performance.count()

    (
        category_performance.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.atlas_gold.category_performance")
    )

    write_audit(
        run_id=pipeline_run_id,
        batch_id=batch_id,
        stage="GOLD_CATEGORY_PERFORMANCE",
        status="SUCCESS",
        started_at=category_started_at,
        records_processed=category_count
    )

    print("Gold category performance created.")
    print(f"Records processed: {category_count}")

except Exception:
    try:
        write_audit(
            run_id=pipeline_run_id,
            batch_id=batch_id,
            stage="GOLD_CATEGORY_PERFORMANCE",
            status="FAILED",
            started_at=category_started_at,
            error_message=traceback.format_exc()[-8000:]
        )
    except Exception as audit_error:
        print(f"Failed to write audit record: {audit_error}")

    raise


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold category performance created.
Records processed: 74


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
categories = spark.table(
    "workspace.atlas_gold.category_performance"
)

print("Number of categories:", categories.count())

categories.select(
    F.sum("items_sold").alias("total_items"),
    F.sum("items_sales_value").alias("total_item_sales"),
    F.sum("freight_value").alias("total_freight")
).show(truncate=False)

display(
    categories
    .orderBy(F.col("items_sales_value").desc_nulls_last())
    .limit(15)
)

Number of categories: 74
+-----------+----------------+-------------+
|total_items|total_item_sales|total_freight|
+-----------+----------------+-------------+
|112650     |13591643.70     |2251909.54   |
+-----------+----------------+-------------+



category,items_sold,order_count,items_sales_value,freight_value,average_item_price,sales_rank
beleza_saude,9670,8836,1258681.34,182566.73,130.163531,1
relogios_presentes,5991,5624,1205005.68,100535.93,201.135984,2
cama_mesa_banho,11115,9417,1036988.68,204693.04,93.296327,3
esporte_lazer,8641,7720,988048.97,168607.51,114.344285,4
informatica_acessorios,7827,6689,911954.32,147318.08,116.513903,5
moveis_decoracao,8334,6449,729762.49,172749.30,87.564494,6
cool_stuff,3796,3632,635290.85,84039.10,167.357969,7
utilidades_domesticas,6964,5884,632248.66,146149.11,90.788148,8
automotivo,4235,3897,592720.11,92664.21,139.957523,9
ferramentas_jardim,4347,3518,485256.46,98962.75,111.630196,10


In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone
import traceback

delivery_started_at = datetime.now(timezone.utc).replace(tzinfo=None)

try:
    orders = spark.table("workspace.atlas_silver.orders")

    delivered_orders = (
        orders
        .filter(
            (F.col("order_status") == "delivered")
            & F.col("order_delivered_customer_date").isNotNull()
            & F.col("order_estimated_delivery_date").isNotNull()
            & F.col("order_purchase_timestamp").isNotNull()
        )
        .withColumn(
            "purchase_date",
            F.to_date("order_purchase_timestamp")
        )
        .withColumn(
            "actual_delivery_days",
            (
                F.col("order_delivered_customer_date").cast("long")
                - F.col("order_purchase_timestamp").cast("long")
            ) / 86400.0
        )
        .withColumn(
            "estimated_delivery_days",
            (
                F.col("order_estimated_delivery_date").cast("long")
                - F.col("order_purchase_timestamp").cast("long")
            ) / 86400.0
        )
        .withColumn(
            "is_late",
            F.col("order_delivered_customer_date")
            > F.col("order_estimated_delivery_date")
        )
    )

    delivery_performance = (
        delivered_orders
        .groupBy("purchase_date")
        .agg(
            F.countDistinct("order_id").alias("delivered_orders"),
            F.round(F.avg("actual_delivery_days"), 2)
            .alias("avg_actual_delivery_days"),
            F.round(F.avg("estimated_delivery_days"), 2)
            .alias("avg_estimated_delivery_days"),
            F.sum(F.col("is_late").cast("int")).alias("late_orders")
        )
        .withColumn(
            "late_delivery_rate_pct",
            F.round(
                F.col("late_orders") / F.col("delivered_orders") * 100,
                2
            )
        )
    )

    delivery_count = delivery_performance.count()

    (
        delivery_performance.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("workspace.atlas_gold.delivery_performance")
    )

    write_audit(
        run_id=pipeline_run_id,
        batch_id=batch_id,
        stage="GOLD_DELIVERY_PERFORMANCE",
        status="SUCCESS",
        started_at=delivery_started_at,
        records_processed=delivery_count
    )

    print("Gold delivery performance table created.")
    print(f"Records processed: {delivery_count}")

except Exception:
    try:
        write_audit(
            run_id=pipeline_run_id,
            batch_id=batch_id,
            stage="GOLD_DELIVERY_PERFORMANCE",
            status="FAILED",
            started_at=delivery_started_at,
            error_message=traceback.format_exc()[-8000:]
        )
    except Exception as audit_error:
        print(f"Failed to write audit record: {audit_error}")

    raise


Gold delivery performance table created.
Records processed: 612


In [0]:
%sql

SELECT
    stage,
    status,
    COUNT(*) AS execution_count,
    SUM(records_processed) AS cumulative_records_processed,
    MAX(created_at) AS latest_execution
FROM workspace.atlas_ops.pipeline_run_audit
GROUP BY stage, status
ORDER BY latest_execution DESC;


stage,status,execution_count,cumulative_records_processed,latest_execution
GOLD_DELIVERY_PERFORMANCE,SUCCESS,2,1224,2026-10-24T22:58:26.439Z
GOLD_CATEGORY_PERFORMANCE,SUCCESS,1,74,2026-10-24T22:57:16.833Z
GOLD_DAILY_SALES,SUCCESS,1,634,2026-10-24T22:57:09.786Z
SILVER_ORDER_ITEM_ENRICHED,SUCCESS,1,112650,2026-10-24T22:53:20.691Z
SILVER_PRODUCTS,SUCCESS,2,65902,2026-10-24T22:52:45.950Z
SILVER_ORDER_ITEMS,SUCCESS,2,225300,2026-10-24T22:52:32.590Z
SILVER_ORDERS,SUCCESS,2,198882,2026-10-24T22:52:15.394Z
BRONZE,SUCCESS,1,245042,2026-10-24T22:47:08.819Z
BRONZE,FAILED,1,245042,2026-10-24T22:45:02.895Z


In [0]:
delivery = spark.table(
    "workspace.atlas_gold.delivery_performance"
)

print("Daily delivery aggregate rows:", delivery.count())

delivery.select(
    F.sum("delivered_orders").alias("total_delivered_orders"),
    F.sum("late_orders").alias("total_late_orders"),
    F.round(F.avg("avg_actual_delivery_days"), 2)
     .alias("mean_of_daily_avg_delivery_days"),
    F.min("late_delivery_rate_pct").alias("min_daily_late_rate_pct"),
    F.max("late_delivery_rate_pct").alias("max_daily_late_rate_pct")
).show(truncate=False)

display(delivery.orderBy("purchase_date").limit(15))

Daily delivery aggregate rows: 612
+----------------------+-----------------+-------------------------------+-----------------------+-----------------------+
|total_delivered_orders|total_late_orders|mean_of_daily_avg_delivery_days|min_daily_late_rate_pct|max_daily_late_rate_pct|
+----------------------+-----------------+-------------------------------+-----------------------+-----------------------+
|96470                 |7826             |12.62                          |0.0                    |100.0                  |
+----------------------+-----------------+-------------------------------+-----------------------+-----------------------+



purchase_date,delivered_orders,avg_actual_delivery_days,avg_estimated_delivery_days,late_orders,late_delivery_rate_pct
2016-09-15,1,54.81,18.49,1,100.0
2016-10-03,7,25.06,46.32,0,0.0
2016-10-04,54,25.76,55.52,2,3.7
2016-10-05,35,20.27,55.19,0,0.0
2016-10-06,41,20.92,57.04,0,0.0
2016-10-07,38,16.1,55.2,0,0.0
2016-10-08,36,14.33,57.71,0,0.0
2016-10-09,20,13.96,53.79,0,0.0
2016-10-10,34,19.23,56.2,1,2.94
2016-12-23,1,4.69,26.03,0,0.0


In [0]:
source = spark.table("workspace.atlas_silver.orders")

expected_delivered = (
    source
    .filter(
        (F.col("order_status") == "delivered")
        & F.col("order_delivered_customer_date").isNotNull()
        & F.col("order_estimated_delivery_date").isNotNull()
        & F.col("order_purchase_timestamp").isNotNull()
    )
    .select("order_id")
    .distinct()
    .count()
)

actual_delivered = delivery.agg(
    F.sum("delivered_orders")
).first()[0]

print("Eligible delivered orders:", expected_delivered)
print("Gold delivered orders:", actual_delivered)
print("Counts reconcile:", expected_delivered == actual_delivered)

Eligible delivered orders: 96470
Gold delivered orders: 96470
Counts reconcile: True


In [0]:
from pyspark.sql import functions as F

daily = spark.table("workspace.atlas_gold.daily_sales")
category = spark.table("workspace.atlas_gold.category_performance")
delivery = spark.table("workspace.atlas_gold.delivery_performance")
enriched = spark.table("workspace.atlas_silver.order_item_enriched")

source_items = enriched.count()
daily_items = daily.agg(F.sum("items_sold")).first()[0]

source_sales = enriched.agg(F.sum("price")).first()[0]
daily_sales_total = daily.agg(F.sum("items_sales_value")).first()[0]

print("DAILY SALES")
print("Items:", source_items, daily_items, source_items == daily_items)
print("Item sales:", source_sales, daily_sales_total,
      source_sales == daily_sales_total)

category_items = category.agg(F.sum("items_sold")).first()[0]
category_sales = category.agg(F.sum("items_sales_value")).first()[0]

print("\nCATEGORY PERFORMANCE")
print("Items:", source_items, category_items,
      source_items == category_items)
print("Item sales:", source_sales, category_sales,
      source_sales == category_sales)

orders = spark.table("workspace.atlas_silver.orders")

eligible = orders.filter(
    (F.col("order_status") == "delivered")
    & F.col("order_delivered_customer_date").isNotNull()
    & F.col("order_estimated_delivery_date").isNotNull()
    & F.col("order_purchase_timestamp").isNotNull()
)

eligible_count = eligible.select("order_id").distinct().count()
gold_delivery_count = delivery.agg(
    F.sum("delivered_orders")
).first()[0]

print("\nDELIVERY PERFORMANCE")
print("Eligible delivered orders:", eligible_count)
print("Gold delivered orders:", gold_delivery_count)
print("Counts reconcile:", eligible_count == gold_delivery_count)

print("\nDaily sales schema:")
daily.printSchema()

print("\nCategory performance schema:")
category.printSchema()

print("\nDelivery performance schema:")
delivery.printSchema()

DAILY SALES
Items: 112650 112650 True
Item sales: 13591643.70 13591643.70 True

CATEGORY PERFORMANCE
Items: 112650 112650 True
Item sales: 13591643.70 13591643.70 True

DELIVERY PERFORMANCE
Eligible delivered orders: 96470
Gold delivered orders: 96470
Counts reconcile: True

Daily sales schema:
root
 |-- purchase_date: date (nullable = true)
 |-- order_count: long (nullable = true)
 |-- customer_count: long (nullable = true)
 |-- delivered_orders: long (nullable = true)
 |-- cancelled_orders: long (nullable = true)
 |-- items_sold: long (nullable = true)
 |-- items_sales_value: decimal(22,2) (nullable = true)
 |-- freight_value: decimal(22,2) (nullable = true)


Category performance schema:
root
 |-- category: string (nullable = true)
 |-- items_sold: long (nullable = true)
 |-- order_count: long (nullable = true)
 |-- items_sales_value: decimal(22,2) (nullable = true)
 |-- freight_value: decimal(22,2) (nullable = true)
 |-- average_item_price: decimal(16,6) (nullable = true)
 |-- sale

In [0]:
%sql
SELECT
    purchase_date,
    items_sales_value,
    order_count
FROM workspace.atlas_gold.daily_sales
ORDER BY purchase_date DESC
LIMIT 10;

purchase_date,items_sales_value,order_count
2018-10-17,0.00,1
2018-10-16,0.00,1
2018-10-03,0.00,1
2018-10-01,0.00,1
2018-09-29,0.00,1
2018-09-26,0.00,1
2018-09-25,0.00,1
2018-09-20,0.00,1
2018-09-17,0.00,1
2018-09-13,0.00,1


In [0]:
%sql
SELECT
    SUM(CASE WHEN purchase_date IS NULL THEN 1 ELSE 0 END)
        AS null_date_rows,
    SUM(CASE WHEN purchase_date IS NOT NULL THEN 1 ELSE 0 END)
        AS dated_rows,
    MAX(purchase_date) AS latest_purchase_date
FROM workspace.atlas_gold.daily_sales;

null_date_rows,dated_rows,latest_purchase_date
0,634,2018-10-17


In [0]:
%sql
SELECT
    purchase_date,
    order_count,
    items_sales_value
FROM workspace.atlas_gold.daily_sales
WHERE purchase_date >= DATE '2018-08-01'
ORDER BY purchase_date;

purchase_date,order_count,items_sales_value
2018-08-01,311,40553.80
2018-08-02,302,38462.26
2018-08-03,314,42075.18
2018-08-04,245,35511.96
2018-08-05,276,35583.76
2018-08-06,372,56427.39
2018-08-07,370,49178.98
2018-08-08,316,43333.07
2018-08-09,289,40040.28
2018-08-10,256,33424.73


In [0]:
%sql
SELECT
    DATE_TRUNC('MONTH', purchase_date) AS purchase_month,
    SUM(order_count) AS total_orders,
    SUM(items_sold) AS total_items_sold,
    SUM(items_sales_value) AS total_item_sales,
    SUM(freight_value) AS total_freight
FROM workspace.atlas_gold.daily_sales
WHERE purchase_date IS NOT NULL
GROUP BY DATE_TRUNC('MONTH', purchase_date)
ORDER BY purchase_month;

purchase_month,total_orders,total_items_sold,total_item_sales,total_freight
2016-09-01T00:00:00.000Z,4,6,267.36,87.39
2016-10-01T00:00:00.000Z,324,363,49507.66,7301.18
2016-12-01T00:00:00.000Z,1,1,10.90,8.72
2017-01-01T00:00:00.000Z,800,955,120312.87,16875.62
2017-02-01T00:00:00.000Z,1780,1951,247303.02,38977.60
2017-03-01T00:00:00.000Z,2682,3000,374344.30,57704.29
2017-04-01T00:00:00.000Z,2404,2684,359927.23,52495.01
2017-05-01T00:00:00.000Z,3700,4136,506071.14,80119.81
2017-06-01T00:00:00.000Z,3245,3583,433038.60,69924.44
2017-07-01T00:00:00.000Z,4026,4519,498031.48,86940.14


In [0]:
%sql
SELECT
    order_status,
    COUNT(*) AS order_count
FROM workspace.atlas_silver.orders
GROUP BY order_status
ORDER BY order_count DESC;

order_status,order_count
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [0]:
%sql
SELECT
    COUNT(*) AS daily_rows,
    SUM(order_count) AS summed_daily_orders,
    SUM(items_sold) AS total_items_sold,
    ROUND(SUM(items_sales_value), 2) AS total_item_sales,
    ROUND(SUM(freight_value), 2) AS total_freight,
    SUM(cancelled_orders) AS cancelled_orders
FROM workspace.atlas_gold.daily_sales;

daily_rows,summed_daily_orders,total_items_sold,total_item_sales,total_freight,cancelled_orders
634,99441,112650,13591643.70,2251909.54,625


In [0]:
%sql
SELECT
    COUNT(*) AS order_rows,
    COUNT(DISTINCT order_id) AS distinct_orders,
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END)
        AS canceled_orders
FROM workspace.atlas_silver.orders;

order_rows,distinct_orders,canceled_orders
99441,99441,625


In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    SUM(CASE
        WHEN order_purchase_timestamp IS NULL THEN 1
        ELSE 0
    END) AS null_purchase_timestamps,
    SUM(CASE
        WHEN order_purchase_timestamp IS NOT NULL THEN 1
        ELSE 0
    END) AS orders_with_purchase_timestamp
FROM workspace.atlas_silver.orders;

total_orders,null_purchase_timestamps,orders_with_purchase_timestamp
99441,0,99441


In [0]:
%sql
SELECT
    order_status,
    COUNT(*) AS orders,
    SUM(CASE
        WHEN order_purchase_timestamp IS NULL THEN 1
        ELSE 0
    END) AS null_purchase_timestamps
FROM workspace.atlas_silver.orders
GROUP BY order_status
ORDER BY orders DESC;

order_status,orders,null_purchase_timestamps
delivered,96478,0
shipped,1107,0
canceled,625,0
unavailable,609,0
invoiced,314,0
processing,301,0
created,5,0
approved,2,0


In [0]:
%sql
SELECT
    SUM(cancelled_orders) AS gold_cancelled_orders
FROM workspace.atlas_gold.daily_sales;

gold_cancelled_orders
625


In [0]:
%sql
SELECT
    COUNT(*) AS total_cancelled,
    SUM(
        CASE
            WHEN order_purchase_timestamp IS NULL THEN 1
            ELSE 0
        END
    ) AS cancelled_with_null_date,
    SUM(
        CASE
            WHEN order_purchase_timestamp IS NOT NULL THEN 1
            ELSE 0
        END
    ) AS cancelled_with_date
FROM workspace.atlas_silver.orders
WHERE order_status = 'canceled';

total_cancelled,cancelled_with_null_date,cancelled_with_date
625,0,625


In [0]:
%sql
SELECT
    SUM(cancelled_orders) AS gold_cancelled_orders,
    COUNT(*) AS gold_date_rows,
    MIN(purchase_date) AS first_date,
    MAX(purchase_date) AS last_date
FROM workspace.atlas_gold.daily_sales;

gold_cancelled_orders,gold_date_rows,first_date,last_date
625,634,2016-09-04,2018-10-17


In [0]:
%sql
WITH source_cancelled AS (
    SELECT
        order_id,
        CAST(order_purchase_timestamp AS DATE) AS purchase_date
    FROM workspace.atlas_silver.orders
    WHERE order_status = 'canceled'
),
gold_by_date AS (
    SELECT
        purchase_date,
        cancelled_orders
    FROM workspace.atlas_gold.daily_sales
)
SELECT
    s.purchase_date,
    COUNT(*) AS source_cancelled_orders,
    COALESCE(g.cancelled_orders, 0) AS gold_cancelled_orders,
    COUNT(*) - COALESCE(g.cancelled_orders, 0) AS difference
FROM source_cancelled s
LEFT JOIN gold_by_date g
    ON s.purchase_date = g.purchase_date
GROUP BY s.purchase_date, g.cancelled_orders
HAVING COUNT(*) <> COALESCE(g.cancelled_orders, 0)
ORDER BY s.purchase_date;

purchase_date,source_cancelled_orders,gold_cancelled_orders,difference


In [0]:
%sql
SELECT
    SUM(cancelled_orders) AS gold_cancelled_orders,
    SUM(order_count) AS gold_daily_orders,
    SUM(items_sold) AS gold_items_sold,
    ROUND(SUM(items_sales_value), 2) AS gold_item_sales
FROM workspace.atlas_gold.daily_sales;

gold_cancelled_orders,gold_daily_orders,gold_items_sold,gold_item_sales
625,99441,112650,13591643.70


In [0]:
%sql
SELECT
    COUNT(*) AS cancelled_orders_in_silver,
    COUNT(e.order_id) AS cancelled_orders_with_enriched_rows,
    COUNT(DISTINCT e.order_id) AS distinct_cancelled_orders_with_enriched_rows
FROM workspace.atlas_silver.orders o
LEFT JOIN (
    SELECT DISTINCT order_id
    FROM workspace.atlas_silver.order_item_enriched
) e
    ON o.order_id = e.order_id
WHERE o.order_status = 'canceled';

cancelled_orders_in_silver,cancelled_orders_with_enriched_rows,distinct_cancelled_orders_with_enriched_rows
625,461,461


In [0]:
%sql
SELECT
    SUM(order_count) AS total_orders,
    SUM(cancelled_orders) AS cancelled_orders,
    SUM(delivered_orders) AS delivered_orders,
    SUM(items_sold) AS items_sold,
    ROUND(SUM(items_sales_value), 2) AS item_sales_value
FROM workspace.atlas_gold.daily_sales;

total_orders,cancelled_orders,delivered_orders,items_sold,item_sales_value
99441,625,96478,112650,13591643.70


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(avg_actual_delivery_days) AS actual_non_null,
    COUNT(avg_estimated_delivery_days) AS estimated_non_null,
    MIN(purchase_date) AS first_date,
    MAX(purchase_date) AS last_date
FROM workspace.atlas_gold.delivery_performance;

total_rows,actual_non_null,estimated_non_null,first_date,last_date
612,612,612,2016-09-15,2018-08-29


In [0]:
%sql
SELECT 'bronze_orders' AS table_name, COUNT(*) AS row_count
FROM workspace.atlas_bronze.orders

UNION ALL

SELECT 'bronze_items', COUNT(*)
FROM workspace.atlas_bronze.order_items

UNION ALL

SELECT 'bronze_products', COUNT(*)
FROM workspace.atlas_bronze.products

UNION ALL

SELECT 'silver_orders', COUNT(*)
FROM workspace.atlas_silver.orders

UNION ALL

SELECT 'silver_items', COUNT(*)
FROM workspace.atlas_silver.order_items

UNION ALL

SELECT 'silver_products', COUNT(*)
FROM workspace.atlas_silver.products

UNION ALL

SELECT 'silver_order_item_enriched', COUNT(*)
FROM workspace.atlas_silver.order_item_enriched;

table_name,row_count
bronze_orders,99441
bronze_items,112650
bronze_products,32951
silver_orders,99441
silver_items,112650
silver_products,32951
silver_order_item_enriched,112650


In [0]:
%sql
SELECT
    'daily_sales' AS table_name,
    COUNT(*) AS row_count
FROM workspace.atlas_gold.daily_sales

UNION ALL

SELECT
    'category_performance',
    COUNT(*)
FROM workspace.atlas_gold.category_performance

UNION ALL

SELECT
    'delivery_performance',
    COUNT(*)
FROM workspace.atlas_gold.delivery_performance;

table_name,row_count
daily_sales,634
category_performance,74
delivery_performance,612


In [0]:
%sql
SELECT
    SUM(items_sold) AS total_items_sold,
    ROUND(SUM(items_sales_value), 2) AS total_item_sales,
    SUM(cancelled_orders) AS total_cancelled_orders,
    SUM(delivered_orders) AS total_delivered_orders
FROM workspace.atlas_gold.daily_sales;

total_items_sold,total_item_sales,total_cancelled_orders,total_delivered_orders
112650,13591643.70,625,96478


In [0]:
from pyspark.sql import functions as F

daily_sales = spark.table("workspace.atlas_gold.daily_sales")

metrics = daily_sales.agg(
    F.sum("items_sold").alias("items_sold"),
    F.round(F.sum("items_sales_value"), 2).alias("item_sales"),
    F.sum("cancelled_orders").alias("cancelled_orders"),
    F.sum("delivered_orders").alias("delivered_orders"),
).first()

expected_metrics = {
    "items_sold": 112650,
    "item_sales": 13591643.70,
    "cancelled_orders": 625,
    "delivered_orders": 96478,
}

failures = []

for metric, expected in expected_metrics.items():
    actual = metrics[metric]
    passed = (
        actual is not None
        and abs(float(actual) - expected) < 0.01
    )

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{metric} | expected={expected}, actual={actual}"
    )

    if not passed:
        failures.append(metric)

if failures:
    raise ValueError(f"Gold metric checks failed: {failures}")

print("All Gold metric checks passed.")

PASS | items_sold | expected=112650, actual=112650
PASS | item_sales | expected=13591643.7, actual=13591643.70
PASS | cancelled_orders | expected=625, actual=625
PASS | delivered_orders | expected=96478, actual=96478
All Gold metric checks passed.


In [0]:
if failures:
    raise ValueError(f"Data-quality checks failed: {failures}")

print("All data-quality checks passed.")

All data-quality checks passed.


In [0]:
orders = spark.table("workspace.atlas_silver.orders")
items = spark.table("workspace.atlas_silver.order_items")
products = spark.table("workspace.atlas_silver.products")
enriched = spark.table("workspace.atlas_silver.order_item_enriched")

failures = []

def check(name, passed, details):
    status = "PASS" if passed else "FAIL"
    print(f"{status} | {name} | {details}")
    if not passed:
        failures.append(name)

duplicate_orders = (
    orders.groupBy("order_id")
    .count()
    .filter("count > 1")
    .count()
)

check(
    "Unique order IDs",
    duplicate_orders == 0,
    f"duplicate groups={duplicate_orders}"
)

duplicate_items = (
    items.groupBy("order_id", "order_item_id")
    .count()
    .filter("count > 1")
    .count()
)

check(
    "Unique order-item keys",
    duplicate_items == 0,
    f"duplicate groups={duplicate_items}"
)

missing_orders = (
    items.select("order_id").distinct()
    .join(
        orders.select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
    .count()
)

check(
    "Order join integrity",
    missing_orders == 0,
    f"unmatched order IDs={missing_orders}"
)

item_product_ids = (
    items.filter("product_id IS NOT NULL")
    .select("product_id")
    .distinct()
)

missing_products = (
    item_product_ids
    .join(
        products.select("product_id").distinct(),
        on="product_id",
        how="left_anti"
    )
    .count()
)

check(
    "Product join integrity",
    missing_products == 0,
    f"unmatched product IDs={missing_products}"
)

item_count = items.count()
enriched_count = enriched.count()

check(
    "Enrichment row preservation",
    enriched_count == item_count,
    f"items={item_count}, enriched={enriched_count}"
)

if failures:
    raise ValueError(f"Data-quality checks failed: {failures}")

print("All key and join-integrity checks passed.")

PASS | Unique order IDs | duplicate groups=0
PASS | Unique order-item keys | duplicate groups=0
PASS | Order join integrity | unmatched order IDs=0
PASS | Product join integrity | unmatched product IDs=0
PASS | Enrichment row preservation | items=112650, enriched=112650
All key and join-integrity checks passed.


In [0]:
from pyspark.sql import functions as F

items = spark.table("workspace.atlas_silver.order_items")
orders = spark.table("workspace.atlas_silver.orders")

failures = []

def check(name, passed, details):
    status = "PASS" if passed else "FAIL"
    print(f"{status} | {name} | {details}")
    if not passed:
        failures.append(name)

negative_prices = items.filter(F.col("price") < 0).count()

check(
    "Non-negative item prices",
    negative_prices == 0,
    f"negative-price rows={negative_prices}"
)

negative_freight = items.filter(F.col("freight_value") < 0).count()

check(
    "Non-negative freight values",
    negative_freight == 0,
    f"negative-freight rows={negative_freight}"
)

null_order_ids = orders.filter(F.col("order_id").isNull()).count()

check(
    "Non-null order IDs",
    null_order_ids == 0,
    f"null order IDs={null_order_ids}"
)

null_purchase_dates = orders.filter(
    F.col("order_purchase_timestamp").isNull()
).count()

check(
    "Non-null purchase timestamps",
    null_purchase_dates == 0,
    f"null purchase timestamps={null_purchase_dates}"
)

if failures:
    raise ValueError(f"Business-rule checks failed: {failures}")

print("All business-rule checks passed.")

PASS | Non-negative item prices | negative-price rows=0
PASS | Non-negative freight values | negative-freight rows=0
PASS | Non-null order IDs | null order IDs=0
PASS | Non-null purchase timestamps | null purchase timestamps=0
All business-rule checks passed.


In [0]:
spark.table("workspace.atlas_gold.daily_sales").printSchema()

root
 |-- purchase_date: date (nullable = true)
 |-- order_count: long (nullable = true)
 |-- customer_count: long (nullable = true)
 |-- delivered_orders: long (nullable = true)
 |-- cancelled_orders: long (nullable = true)
 |-- items_sold: long (nullable = true)
 |-- items_sales_value: decimal(22,2) (nullable = true)
 |-- freight_value: decimal(22,2) (nullable = true)



In [0]:
from pyspark.sql import functions as F

silver_items = spark.table(
    "workspace.atlas_silver.order_items"
)

gold_sales = spark.table(
    "workspace.atlas_gold.daily_sales"
)

silver_totals = silver_items.agg(
    F.count("*").alias("items_sold"),
    F.round(F.sum("price"), 2).alias("items_sales_value")
).first()

gold_totals = gold_sales.agg(
    F.sum("items_sold").alias("items_sold"),
    F.round(F.sum("items_sales_value"), 2).alias("items_sales_value")
).first()

failures = []

def reconcile(metric, silver_value, gold_value):
    passed = (
        silver_value is not None
        and gold_value is not None
        and abs(float(silver_value) - float(gold_value)) < 0.01
    )

    print(
        f"{'PASS' if passed else 'FAIL'} | {metric} | "
        f"Silver={silver_value}, Gold={gold_value}"
    )

    if not passed:
        failures.append(metric)

reconcile(
    "Item count",
    silver_totals["items_sold"],
    gold_totals["items_sold"]
)

reconcile(
    "Item sales value",
    silver_totals["items_sales_value"],
    gold_totals["items_sales_value"]
)

if failures:
    raise ValueError(f"Silver-to-Gold reconciliation failed: {failures}")

print("Silver-to-Gold reconciliation passed.")

PASS | Item count | Silver=112650, Gold=112650
PASS | Item sales value | Silver=13591643.70, Gold=13591643.70
Silver-to-Gold reconciliation passed.


In [0]:
from pyspark.sql import functions as F

silver_orders = spark.table("workspace.atlas_silver.orders")
gold_sales = spark.table("workspace.atlas_gold.daily_sales")

silver_totals = silver_orders.agg(
    F.countDistinct("order_id").alias("order_count"),
    F.countDistinct("customer_id").alias("customer_count"),
    F.sum(
        F.when(F.col("order_status") == "canceled", 1).otherwise(0)
    ).alias("cancelled_orders"),
    F.sum(
        F.when(F.col("order_status") == "delivered", 1).otherwise(0)
    ).alias("delivered_orders")
).first()

gold_totals = gold_sales.agg(
    F.sum("order_count").alias("order_count"),
    F.sum("customer_count").alias("customer_count"),
    F.sum("cancelled_orders").alias("cancelled_orders"),
    F.sum("delivered_orders").alias("delivered_orders")
).first()

failures = []

for metric in [
    "order_count",
    "customer_count",
    "cancelled_orders",
    "delivered_orders"
]:
    silver_value = silver_totals[metric]
    gold_value = gold_totals[metric]

    passed = silver_value == gold_value

    print(
        f"{'PASS' if passed else 'FAIL'} | {metric} | "
        f"Silver={silver_value}, Gold={gold_value}"
    )

    if not passed:
        failures.append(metric)

if failures:
    raise ValueError(f"Order-level reconciliation failed: {failures}")

print("Order-level reconciliation passed.")

PASS | order_count | Silver=99441, Gold=99441
PASS | customer_count | Silver=99441, Gold=99441
PASS | cancelled_orders | Silver=625, Gold=625
PASS | delivered_orders | Silver=96478, Gold=96478
Order-level reconciliation passed.


In [0]:
%sql

CREATE OR REPLACE VIEW workspace.atlas_ops.latest_stage_status AS
WITH ranked AS (
    SELECT
        run_id,
        batch_id,
        pipeline_name,
        stage,
        status,
        started_at,
        completed_at,
        source_file_count,
        records_processed,
        error_message,
        created_at,
        ROW_NUMBER() OVER (
            PARTITION BY stage
            ORDER BY created_at DESC
        ) AS rn
    FROM workspace.atlas_ops.pipeline_run_audit
)
SELECT
    run_id,
    batch_id,
    pipeline_name,
    stage,
    status,
    started_at,
    completed_at,
    source_file_count,
    records_processed,
    error_message,
    created_at
FROM ranked
WHERE rn = 1;


In [0]:
%sql
SELECT *
FROM workspace.atlas_ops.latest_stage_status
ORDER BY created_at DESC;

run_id,batch_id,pipeline_name,stage,status,started_at,completed_at,source_file_count,records_processed,error_message,created_at
91feab5c-0ad2-4482-b709-1c50f5c3d449,atlas_refresh_20261024_224621,Project Atlas - Retail Data Pipeline,GOLD_DELIVERY_PERFORMANCE,SUCCESS,2026-10-24T22:58:23.390Z,2026-10-24T22:58:26.439Z,0,612,null,2026-10-24T22:58:26.439Z
91feab5c-0ad2-4482-b709-1c50f5c3d449,atlas_refresh_20261024_224621,Project Atlas - Retail Data Pipeline,GOLD_CATEGORY_PERFORMANCE,SUCCESS,2026-10-24T22:57:13.507Z,2026-10-24T22:57:16.833Z,0,74,null,2026-10-24T22:57:16.833Z
91feab5c-0ad2-4482-b709-1c50f5c3d449,atlas_refresh_20261024_224621,Project Atlas - Retail Data Pipeline,GOLD_DAILY_SALES,SUCCESS,2026-10-24T22:57:05.027Z,2026-10-24T22:57:09.786Z,0,634,null,2026-10-24T22:57:09.786Z
91feab5c-0ad2-4482-b709-1c50f5c3d449,atlas_refresh_20261024_224621,Project Atlas - Retail Data Pipeline,SILVER_ORDER_ITEM_ENRICHED,SUCCESS,2026-10-24T22:53:16.017Z,2026-10-24T22:53:20.691Z,0,112650,null,2026-10-24T22:53:20.691Z
91feab5c-0ad2-4482-b709-1c50f5c3d449,atlas_refresh_20261024_224621,Project Atlas - Retail Data Pipeline,SILVER_PRODUCTS,SUCCESS,2026-10-24T22:52:43.082Z,2026-10-24T22:52:45.950Z,0,32951,null,2026-10-24T22:52:45.950Z
91feab5c-0ad2-4482-b709-1c50f5c3d449,atlas_refresh_20261024_224621,Project Atlas - Retail Data Pipeline,SILVER_ORDER_ITEMS,SUCCESS,2026-10-24T22:52:29.242Z,2026-10-24T22:52:32.590Z,0,112650,null,2026-10-24T22:52:32.590Z
91feab5c-0ad2-4482-b709-1c50f5c3d449,atlas_refresh_20261024_224621,Project Atlas - Retail Data Pipeline,SILVER_ORDERS,SUCCESS,2026-10-24T22:52:12.388Z,2026-10-24T22:52:15.394Z,0,99441,null,2026-10-24T22:52:15.394Z
91feab5c-0ad2-4482-b709-1c50f5c3d449,atlas_refresh_20261024_224621,Project Atlas - Retail Data Pipeline,BRONZE,SUCCESS,2026-10-24T22:46:17.230Z,2026-10-24T22:47:08.819Z,3,245042,null,2026-10-24T22:47:08.819Z


In [0]:
%sql

CREATE OR REPLACE VIEW workspace.atlas_ops.pipeline_health AS
SELECT
    COUNT(*) AS stages_with_audit,
    SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS successful_stages,
    SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) AS failed_stages,
    SUM(CASE WHEN status NOT IN ('SUCCESS', 'FAILED') THEN 1 ELSE 0 END) AS other_stages,
    MAX(created_at) AS latest_audit_at
FROM workspace.atlas_ops.latest_stage_status;


In [0]:
%sql
SELECT *
FROM workspace.atlas_ops.pipeline_health;

stages_with_audit,successful_stages,failed_stages,other_stages,latest_audit_at
8,8,0,0,2026-10-24T22:58:26.439Z


In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

manifest_table = "workspace.atlas_bronze.file_manifest"
metadata_df = spark.createDataFrame(processed_metadata)

metadata_df = metadata_df.withColumn(
    "file_status",
    F.lit("PROCESSED")
).withColumn(
    "processed_at",
    F.current_timestamp()
)

(
    DeltaTable.forName(spark, manifest_table)
    .alias("target")
    .merge(
        metadata_df.alias("source"),
        "target.source_path = source.source_path"
    )
    .whenMatchedUpdate(set={
        "file_status": "source.file_status",
        "processed_at": "source.processed_at",
        "file_size_bytes": "source.file_size_bytes",
        "last_modified": "source.last_modified"
    })
    .whenNotMatchedInsert(values={
        "source_path": "source.source_path",
        "file_status": "source.file_status",
        "processed_at": "source.processed_at",
        "file_size_bytes": "source.file_size_bytes",
        "last_modified": "source.last_modified"
    })
    .execute()
)

print(
    f"Manifest updated successfully for "
    f"{len(processed_metadata)} source files."
)


Manifest updated successfully for 3 source files.


In [0]:
%sql
SELECT
    source_path,
    file_status,
    file_size_bytes,
    last_modified,
    processed_at
FROM workspace.atlas_bronze.file_manifest
ORDER BY source_path;

source_path,file_status,file_size_bytes,last_modified,processed_at
s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,PROCESSED,15438671,2026-10-24T12:44:02.000Z,2026-10-24T22:24:52.257Z
s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,PROCESSED,17654914,2026-10-24T12:43:54.000Z,2026-10-24T22:24:52.257Z
s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,PROCESSED,2379446,2026-10-24T12:45:19.000Z,2026-10-24T22:24:52.257Z


In [0]:
%sql
SELECT
    source_path,
    file_status,
    processed_at,
    file_size_bytes,
    last_modified
FROM workspace.atlas_bronze.file_manifest
ORDER BY source_path;

source_path,file_status,processed_at,file_size_bytes,last_modified
s3://project-atlas-olist-20261024-180312/raw/olist/order_items/olist_order_items_dataset.csv,PROCESSED,2026-10-24T22:24:52.257Z,15438671,2026-10-24T12:44:02.000Z
s3://project-atlas-olist-20261024-180312/raw/olist/orders/olist_orders_dataset.csv,PROCESSED,2026-10-24T22:24:52.257Z,17654914,2026-10-24T12:43:54.000Z
s3://project-atlas-olist-20261024-180312/raw/olist/products/olist_products_dataset.csv,PROCESSED,2026-10-24T22:24:52.257Z,2379446,2026-10-24T12:45:19.000Z
